# Customer RFM Segmentation - Online Retail II 
## Business Question: Which customers are drifting, and how much revenue is at risk? 

Dataset: Online Retail II - UCI Machine Learning Repository
Transactions from a UK-based online retailer, 2009-2011.

In [1]:
import pandas as pd

df = pd.read_csv(r'C:\Users\44753\Desktop\Portfolio P\P2\online_retail_II.csv')
print(df.shape)
print(df.isnull().sum())

(1067371, 8)
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64


## Step 1: Data Cleaning
Removed rows with missing Customer IDs - no ID means I can't attribute behaviour to a customer.
Removed cancelled orders and zero/negative quantity and price rows - these make the spend calculations inaccurate.
Created a TotalValue column (Quantity × Price) as the revenue metric per line item.

In [2]:
df = df.dropna(subset=['Customer ID'])
df['Customer ID'] = df['Customer ID'].astype(int)
df['Description'] = df['Description'].str.title().str.strip()
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]
df['TotalValue'] = df['Quantity'] * df['Price']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(df.shape)

(805549, 9)


## Step 2: Snapshot Date
RFM recency is calculated using a fixed reference date - the snapshot date.
I use 2011-12-11 (one day after the dataset ends) so recency reflects actual customer behaviour and isn’t affected by when the analysis is performed.

In [3]:
snapshot_date = pd.Timestamp('2011-12-11')
print(f"Snapshot date: {snapshot_date.date()}")
print(f"Latest invoice: {df['InvoiceDate'].max().date()}")

Snapshot date: 2011-12-11
Latest invoice: 2011-12-09


## Step 3: RFM Calculation
- Recency: Number of days between the customer’s last purchase and the snapshot date. Lower values mean more recent activity.
- Frequency: Number of unique invoices. Higher values mean a more engaged customer.
- Monetary: Total amount spent across all transactions.

In [4]:
rfm = df.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('TotalValue', 'sum')
).reset_index()

print(rfm.head())
print(rfm.shape)

   Customer ID  Recency  Frequency  Monetary
0        12346      326         12  77556.46
1        12347        3          8   5633.32
2        12348       76          5   2019.40
3        12349       19          4   4428.69
4        12350      311          1    334.40
(5878, 4)


## Step 4: RFM Scoring
Each metric is scored 1-4 by splitting customers into groups using pd.qcut.
Recency is inverted - a customer who purchased recently gets a higher score despite having a lower recency value.
Frequency uses 3 groups instead of 4 because the data didn't split cleanly into 4 equal groups.

In [5]:
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=4, labels=[4,3,2,1], duplicates='drop')
rfm['F_Score'] = pd.qcut(rfm['Frequency'], q=3, labels=[1,2,3], duplicates='drop')
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=4, labels=[1,2,3,4], duplicates='drop')

rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

print(rfm.head())

   Customer ID  Recency  Frequency  Monetary R_Score F_Score M_Score RFM_Score
0        12346      326         12  77556.46       2       3       4       234
1        12347        3          8   5633.32       4       3       4       434
2        12348       76          5   2019.40       3       2       3       323
3        12349       19          4   4428.69       4       2       4       424
4        12350      311          1    334.40       2       1       1       211


## Step 5: Segment Assignment
Customers are grouped into five segments based on their R, F, and M scores.
This gives the marketing team a clear picture of who to target - Champions need rewarding,
At Risk customers need re-engagement, and Lost customers may not be worth the cost to recover.

In [6]:
def assign_segment(row):
    r, f, m = int(row['R_Score']), int(row['F_Score']), int(row['M_Score'])
    if r == 4 and f >= 2 and m >= 3:
        return 'Champions'
    elif r >= 3 and m >= 3:
        return 'Loyal'
    elif r <= 2 and f >= 2 and m >= 2:
        return 'At Risk'
    elif r <= 2 and f == 1 and m == 1:
        return 'Lost'
    else:
        return 'Needs Attention'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)
print(rfm['Segment'].value_counts())

Segment
Needs Attention    1728
Lost               1090
Champions          1065
At Risk            1044
Loyal               951
Name: count, dtype: int64


## Step 6: Export
Rounding Monetary values to 2 decimal places for clean output.
Exporting one row per customer with all RFM scores, segment label, and total revenue to CSV for Tableau.

In [7]:
rfm['Monetary'] = rfm['Monetary'].round(2)
rfm['Total Revenue'] = rfm['Monetary']

rfm.to_csv(r'C:\Users\44753\Desktop\Portfolio P\P2\rfm_output.csv', index=False)
print("Exported successfully")
print(rfm.shape)

Exported successfully
(5878, 10)
